In [11]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from models.ArtistPopularityModel import ArtistPopularityModel

In [12]:
playlist_metadata = pd.read_parquet("data/original/playlist_metadata.parquet")
playlist_contents = pd.read_parquet("data/original/playlist_contents.parquet")
track_metadata = pd.read_parquet("data/original/track_metadata.parquet")

challenge_metadata = pd.read_parquet("data/challenge/playlist_metadata.parquet")
challenge_tracks = pd.read_parquet("data/challenge/playlist_contents.parquet")

In [13]:
# apply same preprocessing to challenge data that exists on train data
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(challenge_metadata["name"].tolist(), show_progress_bar=True)
challenge_metadata["title_bert_embeddings"] = list(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [14]:
model = ArtistPopularityModel()

In [15]:
model.train(playlist_metadata, playlist_contents, track_metadata)

In [21]:
predictions = model.predict(challenge_metadata, challenge_tracks, track_metadata)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [23]:
# check not recommending something in the playlist already
predictions.merge(challenge_tracks, on=["pid", "track_uri"])

,pid,track_uri,prediction_num,position


In [17]:
# readd the track uri prefix
predictions["track_uri"] = "spotify:track:" + predictions["track_uri"]

In [18]:
import gzip

def save_submission(predictions, output_path="submission.csv.gz"):
    # Sort by pid and prediction_num to ensure correct order
    predictions = predictions.sort_values(["pid", "prediction_num"])
    
    lines = []
    lines.append("team_info, logan_costello, logancostello2022@gmail.com")
    
    for pid, group in predictions.groupby("pid"):
        track_uris = group["track_uri"].tolist()
        line = str(pid) + ", " + ", ".join(track_uris)
        lines.append(line)
    
    content = "\n".join(lines) + "\n"
    
    with gzip.open(output_path, "wt", encoding="utf-8") as f:
        f.write(content)

save_submission(predictions)

In [19]:
predictions.groupby("pid")["track_uri"].apply(lambda x: x.duplicated().sum()).sort_values(ascending=False).head(20)

pid
1000000    0
1011160    0
1011144    0
1011146    0
1011147    0
1011150    0
1011152    0
1011155    0
1011159    0
1011163    0
1011137    0
1011165    0
1011167    0
1011168    0
1011170    0
1011174    0
1011175    0
1011177    0
1011142    0
1011133    0
Name: track_uri, dtype: int64